In [10]:
#!pip install dash
!pip install dash
# Library to Use Dash in Notebooks
# Dash HTML Components Library
from dash import Dash
from dash import html
# Dash Core Components Library
from dash import dcc
from dash import dash_table
from dash.dependencies import Input, Output
# Plotly Express Library for Graphics
import plotly.express as px
import pandas as pd

import sys
!{sys.executable} -m pip install dash-bootstrap-components
import dash_bootstrap_components as dbc

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [67]:

from urllib.request import urlopen
import json
import time
READ_API_KEY = 'YCW7RBBBL5I33G16' # Change this to your READ_API_KEY
CHANNEL_ID = '3312636' # Change this to your CHANNEL_ID
unoR4 = "https://api.thingspeak.com/channels/"+CHANNEL_ID+"/feeds.json?api_key="+READ_API_KEY
TS = urlopen(unoR4) # Change the URL to your read URL
response = TS.read( ) #Request the url
data = json.loads(response) # Get the response of the web server
feeds = data["feeds"] # The data are here, check it.
# It is also possible to read the values from one fields
field1 = "https://api.thingspeak.com/channels/"+CHANNEL_ID+"/fields/1.json?api_key="+READ_API_KEY
TS_1 = urlopen(unoR4) # Change the URL to your read URL
response_1 = TS_1.read( ) #Request the url
data_1 = json.loads(response_1) # Get the response of the web server
feeds_1 = data_1["feeds"] # The data are here, check it.
df = pd.DataFrame(feeds_1)
cols = {"created_at": "time", "entry_id": "entry_id", "field1": "temperature", "field2": "humidity", "field3": "motion"}
df.rename(columns=cols, inplace=True)
df.head()
filtered_df = df.loc[df["motion"] == "1.00000"]

In [68]:
"""First tab is a graph of temperature over time
and a table of all the data
slider to set time range
Humidity over time
slider to set time

table that shows time when motion was detected"""

'First tab is a graph of temperature over time\nand a table of all the data\nslider to set time range\nHumidity over time\nslider to set time\n\ntable that shows time when motion was detected'

In [71]:
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

tab1 = dbc.Card(dbc.CardBody([ html.Div([html.H1("iHome"),
                      dcc.Dropdown(
                      id="y1",
                      options=[{'label': 'Temperature', 'value': 'temperature'},
                              {'label': 'Humidity', 'value': 'humidity'}
                              ],
                      value='temperature'),
                      dcc.Graph(id="graph"),
                                        dash_table.DataTable(df.to_dict())])]), className="mt-1")

tab2 = dbc.Card(dbc.CardBody([html.H1("Motion"),
                      dash_table.DataTable(filtered_df.to_dict('records'), [{"name":i, "id":i} for i in filtered_df.columns])]), className="mt-1")

app.layout = dbc.Container([
  dbc.Row([
    dbc.Col([
      dbc.Tabs([
        dbc.Tab(tab1, label="Home Data"),
        dbc.Tab(tab2, label="Motion Detected"),
      ])
    ])
  ])
])

@app.callback(Output(component_id='graph', component_property='figure'),
              Input(component_id='y1', component_property='value'))
def show_value1(dropdown_option):
    dfd = df[["time", dropdown_option]]
    fig = px.line(dfd, x="time", y=dropdown_option)

    return fig


app.run(debug=False, mode='external', port=8055)